# Read me

This notebook uses a NearestNeighbors model with Euclidean distance to recommend 20 tracks based on the profile of 10 input tracks. The profile is built by calculating the mean of audio features of the input tracks. 

In [67]:
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import spotify as spo

# Load dataset

In [82]:
df = pd.read_csv("./raw_data/spotify-tracks-dataset-detailed.csv")
print(f'Loaded {len(df)} rows')

Loaded 114000 rows


# Cleaning dataset
## Removing Comedy genre

In [83]:
len_before = len(df)
df = df[df["track_genre"] != "comedy"].reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} comedy tracks. Left {len_after} rows')

Removed 1000 comedy tracks. Left 113000 rows


## Droping duplicates

In [84]:
len_before = len(df)
df = df.drop_duplicates(subset=["track_id"]).drop_duplicates(subset=["artists", "track_name"]).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} duplicates. Left {len_after} rows')

Removed 32642 duplicates. Left 80358 rows


## Missing values

In [85]:
len_before = len(df)
df = df.dropna(subset=['album_name']).dropna(subset=['track_name']).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} null values. Left {len(df)} rows')

Removed 1 null values. Left 80357 rows


## Scaling

In [86]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

scaler = StandardScaler().set_output(transform="pandas")

X = df[audio_features]

X_scaled = scaler.fit_transform(X)

In [6]:
X_scaled.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.628394,-0.721328,0.298800,0.551483,-0.849230,-0.503538,0.758094,0.928984,-1.142994
1,-0.847891,-1.896382,-1.794228,-0.079182,1.837684,-0.503524,-0.591264,-0.799395,-1.490909
2,-0.744089,-1.127618,-0.297440,-0.273960,-0.313535,-0.503542,-0.507257,-1.366520,-1.529510
3,-1.735968,-2.320198,-2.049645,-0.457392,1.780439,-0.503313,-0.428501,-1.277786,1.987275
4,0.293923,-0.793026,-0.286864,-0.303272,0.466809,-0.503542,-0.686296,-1.185194,-0.074292


In [7]:
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


# Model 1
- NearestNeighbors with Euclidean distance
- Input tracks: Average acoustic profile (Centroid k-NN)

## Train

In [87]:
# NearestNeighbors with Euclidean distance
model = NearestNeighbors(metric="euclidean")

model.fit(X_scaled)

,n_neighbors,5
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'euclidean'
,p,2
,metric_params,None
,n_jobs,None


## Input tracks

In [91]:
# Build an input track set of 10 tracks
input_tracks = X_scaled.sample(n=10)

In [92]:
# Print input track names
input_indices = input_tracks.index.to_list()
df.iloc[input_indices][['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
31812,6i7DyKzkNHXLbRRnMMinVI,Central 3;Pevê Brito;Drops INA,Cordeiro e Leão (Ao Vivo),Cordeiro e Leão - Ao Vivo,gospel
1578,60Lqpl9aw7eWRD8KDjzPwO,El Michels Affair,Sounding Out the City,Yennicita,afrobeat
1129,1JF1nELbG0d3bxg7EPLGNQ,Los Amigos Invisibles,The New Sound of the Venezuelan Gozadera,Cachete a Cachete,afrobeat
38972,2Jf9W7kmoDcy87ZuuaJ9Eo,Zany;Brennan Heart,Rockin Ur Mind,Bang The Bass - CD Version,hardstyle
60307,2VrSkfinwUSNc7uAXiL0FA,ShaSings;Xtreeemit,Let Me Down Slowly / Mai Dhoondhne Ko Zamaney ...,Let Me Down Slowly / Mai Dhoondhne Ko Zamaney ...,pop
23557,53wJkapBo5tF7oVGAIQd3E,Amanajé Sound System;Monkey Jhayam,Akitafya,Viver,dub
22491,2a2SVYLpz0JjnbFvWh6PSL,Wilkinson;Wretch 32;Nu:Logic,Flatline (Nu:Logic Remix),Flatline - Nu:Logic Remix,drum-and-bass
19466,648R7nEhGZIpZ0zVHJteRF,The Astec Mystic,Submerge Vol.1:Detroit Techno,Jaguar,detroit-techno
57314,1JnZHgpyInXF5DEyEGefWw,Dilsinho;Sorriso Maroto,Juntos (Ao Vivo),Prejuízo - Ao Vivo,pagode
10847,1L4ntmBIXfdF6mZSbWhT5D,Fred Cheng,"逆著風 (劇集 ""失憶24小時"" 主題曲)","逆著風 - 劇集 ""大步走"" 主題曲",cantopop


In [93]:
token = spo.get_access_token()
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[input_indices]['track_id'])
spo.display_player(playlist_id)

In [94]:
# Calculate input profile
input_profile = input_tracks.mean(axis=0).to_frame().T

input_profile

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.1751,0.198984,0.390496,-0.263839,-0.190611,-0.03959,0.190298,0.06405,0.634174


## Recommend

In [95]:
# Find 20 neighbors
distances, indices = model.kneighbors(input_profile, n_neighbors=20)

# The model returns an array of indices
# Return the tracks from the original dataframe
recommendations = df.iloc[indices[0]]


In [96]:
# Print recommendations profile
recommendations_indices = recommendations.index.to_list()
recommendations_tracks = X_scaled.iloc[recommendations_indices]

recommendations_profile = recommendations_tracks.mean(axis=0).to_frame().T

recommendations_profile

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.203681,0.250275,0.473327,-0.408472,-0.309941,-0.561037,0.13304,0.141534,0.617812


In [97]:
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[recommendations_indices]['track_id'])
spo.display_player(playlist_id)

In [98]:
recommendations[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
24024,3A7qX2QjDlPnazUsRk5y0M,David Guetta;Justin Bieber,2U (feat. Justin Bieber),2U (feat. Justin Bieber),edm
48615,7iaQV6plfQRead7XdleWM4,Caravan Palace,<I°_°I>,Aftermath,jazz
50606,3CpD9lnEHelRkBtsdktwHy,Luis Fonsi,Ya huele a buñuelos,Let It Snow,latin
51674,5CR87wvKHb3b5Wkm1s66kk,Gersson Edinbaro,Ummaal Koodaatha,Ummaal Koodaatha,malay
23532,0e0UxWGgjXoYAYUFhJgwji,ILLENIUM;Gryffin;Daya,Awake,Feel Good,dub
15294,3JMTWuUH9FTbn2CqhGPLeC,Woesum;Bladee;Yung Lean,Blue Summer,Airwalker,club
24296,0Ev562zA4pmUsBqjxsxxdx,Gryffin;ILLENIUM;Daya,Feel Good (feat. Daya),Feel Good (feat. Daya),edm
13255,0m3DrWZvSJBKBdcJsTmjfy,MASN,Strawberry Huracan,Strawberry Huracan,chill
15798,7rVw4gyhclOgTIrFCtI3ZG,Brett Eldredge,Chillin' It - Mellow Day Country,Bring You Back,country
51769,4QzJOTh94tmzsR1lV6mfML,Sajeer Koppam,"Dilse, Vol. 2",Vannu ninte,malay


# Alternative models
- Model: NearestNeighbors with Manhattan distance
- Input tracks: average of multimodal preferences (Multi-seed k-NN)